# Notebook 13: Two-Phase Model — Binary Classification Evaluation

This notebook provides comprehensive evaluation of the two-phase DR classification model for binary disease/no-disease classification.
It covers ROC curves, confusion matrices, threshold analysis, and subgroup performance analysis.

In [3]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix, 
                             precision_recall_curve, auc as sklearn_auc,
                             accuracy_score, sensitivity_specificity_support,
                             brier_score_loss)
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

BASE_DIR = '.'
FIGURES_DIR = f'{BASE_DIR}/figures'
import os
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete. Loading data...")

ImportError: cannot import name 'sensitivity_specificity_support' from 'sklearn.metrics' (/opt/miniconda3/lib/python3.12/site-packages/sklearn/metrics/__init__.py)

## 1. Load Data

Load the APTOS comprehensive test predictions and evaluation metrics.

In [ ]:
# Load predictions
try:
    predictions_df = pd.read_csv(f'{BASE_DIR}/evaluation/aptos_comprehensive/test_predictions_full.csv')
    print(f"Loaded predictions: {predictions_df.shape[0]} samples")
    print(f"Columns: {predictions_df.columns.tolist()}")
except FileNotFoundError:
    print("ERROR: test_predictions_full.csv not found")
    predictions_df = None

# Extract two-phase columns
if predictions_df is not None:
    y_true = predictions_df['binary_label'].values
    prob_no_tta = predictions_df['prob_twophase_noTTA'].values
    prob_tta = predictions_df['prob_twophase_TTA'].values
    dr_grades = predictions_df['dr_grade'].values
    
    print(f"\nBinary class distribution:")
    print(predictions_df['binary_label'].value_counts().sort_index())
    print(f"\nDR grade distribution:")
    print(predictions_df['dr_grade'].value_counts().sort_index())

## 2. Load Evaluation Metrics JSON

Load comprehensive evaluation metrics including confidence intervals.

In [ ]:
try:
    with open(f'{BASE_DIR}/evaluation/aptos_comprehensive/comprehensive_eval_v2.json', 'r') as f:
        eval_data = json.load(f)
    print("Loaded evaluation metrics")
    print(f"Dataset: {eval_data['dataset']}")
    print(f"Task: {eval_data['task']}")
    print(f"Test size: {eval_data['test_size']}")
except FileNotFoundError:
    print("ERROR: comprehensive_eval_v2.json not found")
    eval_data = None

## 3. ROC Curve Analysis

Plot ROC curves for both no-TTA and TTA variants, with AUC annotations.

In [ ]:
if predictions_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # No-TTA ROC
    fpr_no_tta, tpr_no_tta, _ = roc_curve(y_true, prob_no_tta)
    auc_no_tta = roc_auc_score(y_true, prob_no_tta)
    
    axes[0].plot(fpr_no_tta, tpr_no_tta, linewidth=2.5, color='#2196F3', 
                label=f'Two-Phase (no TTA), AUC = {auc_no_tta:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3)
    axes[0].set_xlabel('False Positive Rate', fontsize=11)
    axes[0].set_ylabel('True Positive Rate', fontsize=11)
    axes[0].set_title('ROC Curve: Two-Phase (no TTA)', fontsize=12, fontweight='bold')
    axes[0].legend(loc='lower right', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([-0.01, 1.01])
    axes[0].set_ylim([-0.01, 1.01])
    
    # TTA ROC
    fpr_tta, tpr_tta, _ = roc_curve(y_true, prob_tta)
    auc_tta = roc_auc_score(y_true, prob_tta)
    
    axes[1].plot(fpr_tta, tpr_tta, linewidth=2.5, color='#2196F3',
                label=f'Two-Phase (TTA), AUC = {auc_tta:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3)
    axes[1].set_xlabel('False Positive Rate', fontsize=11)
    axes[1].set_ylabel('True Positive Rate', fontsize=11)
    axes[1].set_title('ROC Curve: Two-Phase (TTA)', fontsize=12, fontweight='bold')
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([-0.01, 1.01])
    axes[1].set_ylim([-0.01, 1.01])
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"AUC-ROC (no TTA): {auc_no_tta:.6f}")
    print(f"AUC-ROC (TTA): {auc_tta:.6f}")

## 4. Precision-Recall Curve

PR curves show the tradeoff between precision and recall across thresholds.

In [ ]:
if predictions_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # No-TTA PR
    precision_no_tta, recall_no_tta, _ = precision_recall_curve(y_true, prob_no_tta)
    pr_auc_no_tta = sklearn_auc(recall_no_tta, precision_no_tta)
    
    axes[0].plot(recall_no_tta, precision_no_tta, linewidth=2.5, color='#2196F3',
                label=f'Two-Phase (no TTA), AUC-PR = {pr_auc_no_tta:.4f}')
    axes[0].axhline(y=y_true.mean(), color='k', linestyle='--', alpha=0.3, label='Baseline')
    axes[0].set_xlabel('Recall', fontsize=11)
    axes[0].set_ylabel('Precision', fontsize=11)
    axes[0].set_title('Precision-Recall Curve: Two-Phase (no TTA)', fontsize=12, fontweight='bold')
    axes[0].legend(loc='best', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([-0.01, 1.01])
    axes[0].set_ylim([-0.01, 1.01])
    
    # TTA PR
    precision_tta, recall_tta, _ = precision_recall_curve(y_true, prob_tta)
    pr_auc_tta = sklearn_auc(recall_tta, precision_tta)
    
    axes[1].plot(recall_tta, precision_tta, linewidth=2.5, color='#2196F3',
                label=f'Two-Phase (TTA), AUC-PR = {pr_auc_tta:.4f}')
    axes[1].axhline(y=y_true.mean(), color='k', linestyle='--', alpha=0.3, label='Baseline')
    axes[1].set_xlabel('Recall', fontsize=11)
    axes[1].set_ylabel('Precision', fontsize=11)
    axes[1].set_title('Precision-Recall Curve: Two-Phase (TTA)', fontsize=12, fontweight='bold')
    axes[1].legend(loc='best', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([-0.01, 1.01])
    axes[1].set_ylim([-0.01, 1.01])
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_pr_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"AUC-PR (no TTA): {pr_auc_no_tta:.6f}")
    print(f"AUC-PR (TTA): {pr_auc_tta:.6f}")

## 5. Confusion Matrices

2x2 heatmaps showing prediction accuracy for binary classification.

In [ ]:
if predictions_df is not None and eval_data is not None:
    # Get optimal threshold from eval_data
    opt_threshold = eval_data.get('optimal_threshold_twophase', 0.5)
    
    # Predictions at default threshold (0.5) and optimal threshold
    pred_no_tta = (prob_no_tta >= 0.5).astype(int)
    pred_tta = (prob_tta >= 0.5).astype(int)
    pred_tta_opt = (prob_tta >= opt_threshold).astype(int)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # No-TTA confusion matrix
    cm_no_tta = confusion_matrix(y_true, pred_no_tta)
    sns.heatmap(cm_no_tta, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['No DR', 'DR'], yticklabels=['No DR', 'DR'],
                cbar_kws={'label': 'Count'})
    axes[0].set_title('Two-Phase (no TTA)\nThreshold=0.5', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('True Label', fontsize=10)
    axes[0].set_xlabel('Predicted Label', fontsize=10)
    
    # TTA confusion matrix
    cm_tta = confusion_matrix(y_true, pred_tta)
    sns.heatmap(cm_tta, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['No DR', 'DR'], yticklabels=['No DR', 'DR'],
                cbar_kws={'label': 'Count'})
    axes[1].set_title('Two-Phase (TTA)\nThreshold=0.5', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('True Label', fontsize=10)
    axes[1].set_xlabel('Predicted Label', fontsize=10)
    
    # TTA + Optimal Threshold
    cm_tta_opt = confusion_matrix(y_true, pred_tta_opt)
    sns.heatmap(cm_tta_opt, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=['No DR', 'DR'], yticklabels=['No DR', 'DR'],
                cbar_kws={'label': 'Count'})
    axes[2].set_title(f'Two-Phase (TTA + OptThr)\nThreshold={opt_threshold:.4f}', 
                     fontsize=11, fontweight='bold')
    axes[2].set_ylabel('True Label', fontsize=10)
    axes[2].set_xlabel('Predicted Label', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nOptimal threshold for TTA: {opt_threshold:.6f}")

## 6. Threshold Analysis

Load and analyze threshold sweeps showing how metrics change with different decision thresholds.

In [ ]:
try:
    threshold_df = pd.read_csv(f'{BASE_DIR}/evaluation/aptos_comprehensive/threshold_sweep_twophase.csv')
    print(f"Loaded threshold sweep: {len(threshold_df)} thresholds tested")
    print(threshold_df.head(3))
except FileNotFoundError:
    print("ERROR: threshold_sweep_twophase.csv not found")
    threshold_df = None

In [ ]:
if threshold_df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    opt_idx = threshold_df['f1'].idxmax()
    opt_thr = threshold_df.loc[opt_idx, 'threshold']
    
    # Sensitivity and Specificity
    axes[0, 0].plot(threshold_df['threshold'], threshold_df['sensitivity'], 
                    linewidth=2, color='#2196F3', label='Sensitivity')
    axes[0, 0].plot(threshold_df['threshold'], threshold_df['specificity'],
                    linewidth=2, color='#FF9800', label='Specificity')
    axes[0, 0].axvline(opt_thr, color='green', linestyle='--', linewidth=1.5, 
                       label=f'Optimal (F1): {opt_thr:.4f}')
    axes[0, 0].set_xlabel('Threshold', fontsize=10)
    axes[0, 0].set_ylabel('Score', fontsize=10)
    axes[0, 0].set_title('Sensitivity & Specificity vs Threshold', fontsize=11, fontweight='bold')
    axes[0, 0].legend(fontsize=9)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Precision and NPV
    axes[0, 1].plot(threshold_df['threshold'], threshold_df['precision'],
                    linewidth=2, color='#2196F3', label='Precision')
    axes[0, 1].plot(threshold_df['threshold'], threshold_df['npv'],
                    linewidth=2, color='#FF9800', label='NPV')
    axes[0, 1].axvline(opt_thr, color='green', linestyle='--', linewidth=1.5,
                       label=f'Optimal: {opt_thr:.4f}')
    axes[0, 1].set_xlabel('Threshold', fontsize=10)
    axes[0, 1].set_ylabel('Score', fontsize=10)
    axes[0, 1].set_title('Precision & NPV vs Threshold', fontsize=11, fontweight='bold')
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Accuracy and F1
    axes[1, 0].plot(threshold_df['threshold'], threshold_df['accuracy'],
                    linewidth=2, color='#2196F3', label='Accuracy')
    axes[1, 0].plot(threshold_df['threshold'], threshold_df['f1'],
                    linewidth=2, color='#FF9800', label='F1-Score')
    axes[1, 0].axvline(opt_thr, color='green', linestyle='--', linewidth=1.5,
                       label=f'Optimal: {opt_thr:.4f}')
    axes[1, 0].set_xlabel('Threshold', fontsize=10)
    axes[1, 0].set_ylabel('Score', fontsize=10)
    axes[1, 0].set_title('Accuracy & F1 vs Threshold', fontsize=11, fontweight='bold')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Youden's J
    axes[1, 1].plot(threshold_df['threshold'], threshold_df['youden_j'],
                    linewidth=2, color='#2196F3', label='Youden\'s J')
    axes[1, 1].axvline(opt_thr, color='green', linestyle='--', linewidth=1.5,
                       label=f'Optimal: {opt_thr:.4f}')
    axes[1, 1].set_xlabel('Threshold', fontsize=10)
    axes[1, 1].set_ylabel('Youden\'s J', fontsize=10)
    axes[1, 1].set_title('Youden\'s J vs Threshold', fontsize=11, fontweight='bold')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_threshold_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nOptimal threshold (max F1): {opt_thr:.6f}")
    print(f"Accuracy at optimal: {threshold_df.loc[opt_idx, 'accuracy']:.4f}")
    print(f"F1 at optimal: {threshold_df.loc[opt_idx, 'f1']:.4f}")

## 7. Metrics Summary Table

Comprehensive comparison of all evaluation variants.

In [ ]:
if eval_data is not None:
    # Extract metrics for all variants
    metrics_dict = {}
    
    for variant_name, metrics in eval_data['evaluation_levels'].items():
        metrics_dict[variant_name] = {
            'Threshold': metrics.get('Threshold', 'N/A'),
            'AUC-ROC': metrics.get('AUC-ROC', np.nan),
            'AUC-PR': metrics.get('AUC-PR', np.nan),
            'Accuracy': metrics.get('Accuracy', np.nan),
            'Sensitivity': metrics.get('Sensitivity', np.nan),
            'Specificity': metrics.get('Specificity', np.nan),
            'Precision': metrics.get('Precision', np.nan),
            'F1-Score': metrics.get('F1-Score', np.nan),
            'Cohen Kappa': metrics.get('Cohen Kappa', np.nan),
        }
    
    metrics_df = pd.DataFrame(metrics_dict).T
    
    print("\n" + "="*100)
    print("BINARY CLASSIFICATION METRICS SUMMARY")
    print("="*100)
    print(metrics_df.round(6).to_string())
    print("="*100)

## 8. Subgroup Analysis by DR Grade

Performance breakdown across DR severity grades (0-4).

In [ ]:
if predictions_df is not None:
    pred_tta = (prob_tta >= 0.5).astype(int)
    
    # Calculate accuracy per grade
    grade_acc = {}
    for grade in sorted(np.unique(dr_grades)):
        mask = dr_grades == grade
        if mask.sum() > 0:
            acc = (pred_tta[mask] == y_true[mask]).mean()
            grade_acc[f'Grade {grade}'] = acc
    
    fig, ax = plt.subplots(figsize=(10, 6))
    grades = list(grade_acc.keys())
    accuracies = list(grade_acc.values())
    
    bars = ax.bar(grades, accuracies, color='#2196F3', alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{acc:.3f}',
               ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_xlabel('DR Severity Grade', fontsize=11)
    ax.set_title('Binary Classification Accuracy by DR Grade', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_subgroup_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nAccuracy by DR Grade:")
    for grade, acc in grade_acc.items():
        print(f"  {grade}: {acc:.4f}")

## 9. Bootstrap Confidence Intervals

Compute 95% confidence intervals for key metrics using 2000 bootstrap samples.

In [ ]:
if predictions_df is not None:
    n_bootstrap = 2000
    np.random.seed(42)
    
    auc_samples = []
    sensitivity_samples = []
    specificity_samples = []
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(y_true), size=len(y_true), replace=True)
        y_boot = y_true[idx]
        prob_boot = prob_tta[idx]
        
        # AUC
        if len(np.unique(y_boot)) > 1:
            auc_samples.append(roc_auc_score(y_boot, prob_boot))
        
        # Sensitivity and Specificity
        pred_boot = (prob_boot >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_boot, pred_boot).ravel()
        sensitivity_samples.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        specificity_samples.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    
    # Calculate 95% CIs
    auc_ci = (np.percentile(auc_samples, 2.5), np.percentile(auc_samples, 97.5))
    sens_ci = (np.percentile(sensitivity_samples, 2.5), np.percentile(sensitivity_samples, 97.5))
    spec_ci = (np.percentile(specificity_samples, 2.5), np.percentile(specificity_samples, 97.5))
    
    print("\nBootstrap 95% Confidence Intervals (2000 samples):")
    print(f"  AUC-ROC: [{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]")
    print(f"  Sensitivity: [{sens_ci[0]:.4f}, {sens_ci[1]:.4f}]")
    print(f"  Specificity: [{spec_ci[0]:.4f}, {spec_ci[1]:.4f}]")
    
    # Plot distributions
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(auc_samples, bins=40, color='#2196F3', alpha=0.7, edgecolor='black')
    axes[0].axvline(np.mean(auc_samples), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[0].axvline(auc_ci[0], color='green', linestyle='--', linewidth=2, label='95% CI')
    axes[0].axvline(auc_ci[1], color='green', linestyle='--', linewidth=2)
    axes[0].set_xlabel('AUC-ROC', fontsize=10)
    axes[0].set_ylabel('Frequency', fontsize=10)
    axes[0].set_title('Bootstrap AUC Distribution', fontsize=11, fontweight='bold')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].hist(sensitivity_samples, bins=40, color='#2196F3', alpha=0.7, edgecolor='black')
    axes[1].axvline(np.mean(sensitivity_samples), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[1].axvline(sens_ci[0], color='green', linestyle='--', linewidth=2, label='95% CI')
    axes[1].axvline(sens_ci[1], color='green', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Sensitivity', fontsize=10)
    axes[1].set_ylabel('Frequency', fontsize=10)
    axes[1].set_title('Bootstrap Sensitivity Distribution', fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    axes[2].hist(specificity_samples, bins=40, color='#2196F3', alpha=0.7, edgecolor='black')
    axes[2].axvline(np.mean(specificity_samples), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[2].axvline(spec_ci[0], color='green', linestyle='--', linewidth=2, label='95% CI')
    axes[2].axvline(spec_ci[1], color='green', linestyle='--', linewidth=2)
    axes[2].set_xlabel('Specificity', fontsize=10)
    axes[2].set_ylabel('Frequency', fontsize=10)
    axes[2].set_title('Bootstrap Specificity Distribution', fontsize=11, fontweight='bold')
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_bootstrap_ci.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Calibration Plot

Reliability diagram showing the relationship between predicted probability and actual positive rate.

In [ ]:
if predictions_df is not None:
    # Create probability bins
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    bin_true_rate = []
    bin_pred_rate = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (prob_tta >= bin_edges[i]) & (prob_tta < bin_edges[i+1])
        if mask.sum() > 0:
            bin_true_rate.append(y_true[mask].mean())
            bin_pred_rate.append(prob_tta[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_true_rate.append(np.nan)
            bin_pred_rate.append(np.nan)
            bin_counts.append(0)
    
    # Brier score
    brier = brier_score_loss(y_true, prob_tta)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Perfect calibration line
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
    
    # Calibration curve
    valid_mask = ~np.isnan(bin_pred_rate)
    ax.plot(np.array(bin_pred_rate)[valid_mask], np.array(bin_true_rate)[valid_mask],
           'o-', color='#2196F3', linewidth=2, markersize=8, label='Two-Phase (TTA)',
           markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax.set_ylabel('Fraction of Positives', fontsize=11)
    ax.set_title(f'Calibration Plot - Two-Phase (TTA)\nBrier Score: {brier:.4f}', 
                fontsize=12, fontweight='bold')
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_calibration_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nBrier Score (lower is better): {brier:.6f}")

## 11. Error Analysis

Identify and analyze misclassified cases.

In [ ]:
if predictions_df is not None:
    pred_tta = (prob_tta >= 0.5).astype(int)
    is_correct = (pred_tta == y_true)
    
    misclassified = predictions_df[~is_correct].copy()
    misclassified['pred_prob'] = prob_tta[~is_correct]
    misclassified['error_type'] = misclassified.apply(
        lambda x: 'False Positive' if x['binary_label'] == 0 else 'False Negative',
        axis=1
    )
    
    print(f"\nMisclassification Summary:")
    print(f"  Total misclassified: {len(misclassified)} / {len(predictions_df)} ({len(misclassified)/len(predictions_df)*100:.2f}%)")
    print(f"\nError type breakdown:")
    print(misclassified['error_type'].value_counts())
    
    if len(misclassified) > 0:
        print(f"\nTop 10 misclassified samples (by confidence):")
        display_cols = ['image_id', 'binary_label', 'dr_grade', 'pred_prob', 'error_type']
        print(misclassified.nlargest(10, 'pred_prob')[display_cols].to_string())
    
    # Visualize error distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Error type counts
    error_counts = misclassified['error_type'].value_counts()
    axes[0].bar(error_counts.index, error_counts.values, color=['#FF6B6B', '#4ECDC4'],
               edgecolor='black', linewidth=1.5)
    axes[0].set_ylabel('Count', fontsize=10)
    axes[0].set_title('Error Type Distribution', fontsize=11, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Prediction probability distribution of errors
    fp_probs = misclassified[misclassified['error_type'] == 'False Positive']['pred_prob']
    fn_probs = misclassified[misclassified['error_type'] == 'False Negative']['pred_prob']
    
    axes[1].hist(fp_probs, bins=15, alpha=0.6, label=f'False Positives (n={len(fp_probs)})',
                color='#FF6B6B', edgecolor='black')
    axes[1].hist(fn_probs, bins=15, alpha=0.6, label=f'False Negatives (n={len(fn_probs)})',
                color='#4ECDC4', edgecolor='black')
    axes[1].set_xlabel('Predicted Probability', fontsize=10)
    axes[1].set_ylabel('Count', fontsize=10)
    axes[1].set_title('Prediction Probability of Misclassified Cases', fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/13_error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()